# 🛰️ ISRO Satellite Image — Land Cover Classification
**Project for ISRO Remote Sensing | LISS-IV / Resourcesat Data**

This notebook classifies land cover (Urban, Forest, Water, Agriculture, Barren, Wetlands) from satellite imagery.

**Data Sources:**
- Bhuvan Portal: https://bhuvan.nrsc.gov.in
- Bhoonidhi Portal: https://bhoonidhi.nrsc.gov.in

**Run all cells top to bottom! ▶️**

## 📦 Step 1: Install Libraries

In [ ]:
# Install required libraries
!pip install rasterio folium scikit-learn matplotlib numpy joblib -q
print('✅ All libraries installed!')

## 📥 Step 2: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import os, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
import joblib
import folium
from IPython.display import display, HTML

print('✅ All imports successful!')

## ⚙️ Step 3: Configuration — Land Cover Classes

In [ ]:
LAND_COVER_CLASSES = {
    0: 'Unclassified',
    1: 'Urban / Built-up',
    2: 'Forest / Dense Vegetation',
    3: 'Agriculture / Cropland',
    4: 'Water Bodies',
    5: 'Barren / Rocky Land',
    6: 'Wetlands / Marshy',
}

CLASS_COLORS = {
    0: '#808080',
    1: '#FF4444',
    2: '#228B22',
    3: '#ADFF2F',
    4: '#1E90FF',
    5: '#D2B48C',
    6: '#40E0D0',
}

print('✅ Config loaded! Classes:', list(LAND_COVER_CLASSES.values()))

## 📡 Step 4: Load Data
> **Option A:** Demo with synthetic data (no download needed) — Run cell below as-is
> 
> **Option B:** Use real Bhuvan data — Upload your .tif file to Colab, then uncomment the real loader

In [ ]:
# ════════════════════════════════════════════
# OPTION A: SYNTHETIC DATA (run immediately)
# ════════════════════════════════════════════

def generate_synthetic_satellite_data(height=256, width=256, n_bands=4, seed=42):
    np.random.seed(seed)
    image = np.zeros((n_bands, height, width), dtype=np.float32)
    
    # Urban: top-left
    for b in range(4):
        image[b, :80, :80] = np.random.normal([0.25,0.25,0.30,0.20][b], 0.05, (80,80))
    
    # Forest: top-right (high NIR)
    for b in range(4):
        image[b, :80, 80:] = np.random.normal([0.05,0.15,0.08,0.45][b], 0.03, (80,176))
    
    # Water: bottom-left (low all)
    for b in range(4):
        image[b, 160:, :100] = np.random.normal([0.10,0.08,0.05,0.02][b], 0.02, (96,100))
    
    # Agriculture: center
    for b in range(4):
        image[b, 80:160, 80:176] = np.random.normal([0.08,0.25,0.12,0.38][b], 0.04, (80,96))
    
    # Barren: bottom-right
    for b in range(4):
        image[b, 160:, 100:] = np.random.normal([0.30,0.32,0.35,0.28][b], 0.06, (96,156))
    
    # Wetlands: middle-left
    for b in range(4):
        image[b, 80:160, :80] = np.random.normal([0.12,0.18,0.10,0.15][b], 0.03, (80,80))
    
    return np.clip(image + np.random.normal(0, 0.02, image.shape), 0, 1).astype(np.float32)

def generate_ground_truth(height=256, width=256):
    labels = np.zeros((height, width), dtype=np.uint8)
    labels[:80, :80] = 1
    labels[:80, 80:] = 2
    labels[160:, :100] = 4
    labels[80:160, 80:176] = 3
    labels[160:, 100:] = 5
    labels[80:160, :80] = 6
    return labels

image = generate_synthetic_satellite_data()
true_labels = generate_ground_truth()
height, width = image.shape[1], image.shape[2]
center_lat, center_lon = 20.5937, 78.9629

print(f'✅ Data ready! Image shape: {image.shape}')
print('   Bands: [Blue, Green, Red, NIR]')

In [ ]:
# ════════════════════════════════════════════
# OPTION B: REAL BHUVAN DATA (uncomment & use)
# ════════════════════════════════════════════

# from google.colab import files
# uploaded = files.upload()  # Upload your .tif file
# tif_file = list(uploaded.keys())[0]
#
# with rasterio.open(tif_file) as src:
#     image = src.read().astype(np.float32)
#     bounds = src.bounds
#     for b in range(image.shape[0]):
#         mn, mx = image[b].min(), image[b].max()
#         if mx > mn: image[b] = (image[b]-mn)/(mx-mn)
#     center_lat = (bounds.top + bounds.bottom) / 2
#     center_lon = (bounds.left + bounds.right) / 2
#
# height, width = image.shape[1], image.shape[2]
# true_labels = generate_ground_truth(height, width)  # or load your shapefiles
# print(f'Real image loaded! Shape: {image.shape}')


## 📊 Step 5: Compute Spectral Indices (NDVI, NDWI, NDBI, EVI)

In [ ]:
def compute_spectral_indices(img):
    eps = 1e-8
    blue, green, red, nir = img[0], img[1], img[2], img[3]
    ndvi = (nir - red) / (nir + red + eps)
    ndwi = (green - nir) / (green + nir + eps)
    ndbi = (red - nir) / (red + nir + eps)
    evi  = 2.5*(nir-red)/(nir+6*red-7.5*blue+1+eps)
    return ndvi, ndwi, ndbi, evi

def extract_features(img):
    n_bands, h, w = img.shape
    bands_flat = img.reshape(n_bands, -1).T
    ndvi, ndwi, ndbi, evi = compute_spectral_indices(img)
    eps = 1e-8
    indices_flat = np.stack([ndvi.ravel(), ndwi.ravel(), ndbi.ravel(), evi.ravel()], axis=1)
    r_g = (img[2]/(img[1]+eps)).ravel().reshape(-1,1)
    n_r = (img[3]/(img[2]+eps)).ravel().reshape(-1,1)
    return np.concatenate([bands_flat, indices_flat, r_g, n_r], axis=1)

features = extract_features(image)
ndvi, ndwi, ndbi, evi = compute_spectral_indices(image)

print(f'✅ Features extracted: {features.shape}')

# Visualize indices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.patch.set_facecolor('#0D1117')
titles = ['NDVI (Vegetation)', 'NDWI (Water)', 'NDBI (Built-up)', 'EVI (Enhanced Veg)']
cmaps  = ['RdYlGn', 'Blues', 'RdBu_r', 'Greens']
for ax, data, title, cmap in zip(axes, [ndvi,ndwi,ndbi,evi], titles, cmaps):
    ax.set_facecolor('#0D1117')
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title, color='white', fontweight='bold')
    plt.colorbar(im, ax=ax)
    ax.axis('off')
plt.suptitle('📊 Spectral Indices', color='white', fontsize=14)
plt.tight_layout()
plt.show()

## 🌲 Step 6: Train Random Forest Classifier

In [ ]:
labels_flat = true_labels.ravel()
mask = labels_flat > 0
X, y = features[mask], labels_flat[mask]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=2,
    random_state=42, n_jobs=-1, class_weight='balanced'
)
rf.fit(X_train_s, y_train)

y_pred = rf.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)

class_names = [LAND_COVER_CLASSES[c] for c in np.unique(y_test)]
print(f'✅ Accuracy: {acc*100:.2f}%')
print(classification_report(y_test, y_pred, target_names=class_names))

## 🗺️ Step 7: Classify Full Image & Visualize

In [ ]:
# Classify full image
features_s = scaler.transform(features)
predicted_map = rf.predict(features_s).reshape(height, width)

# Plot
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap_vals = [CLASS_COLORS.get(k,'#000') for k in sorted(CLASS_COLORS.keys())]
cmap = ListedColormap(cmap_vals)
norm = BoundaryNorm(np.arange(-0.5, len(LAND_COVER_CLASSES)+0.5), len(LAND_COVER_CLASSES))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0D1117')

# RGB
rgb = np.stack([image[2],image[1],image[0]], axis=2)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
axes[0].imshow(rgb); axes[0].set_title('RGB Composite', color='white', fontweight='bold'); axes[0].axis('off')
axes[0].set_facecolor('#0D1117')

# Ground Truth
axes[1].imshow(true_labels, cmap=cmap, norm=norm)
axes[1].set_title('Ground Truth', color='white', fontweight='bold'); axes[1].axis('off')
axes[1].set_facecolor('#0D1117')

# Predicted
axes[2].imshow(predicted_map, cmap=cmap, norm=norm)
axes[2].set_title(f'ML Prediction (Acc: {acc*100:.1f}%)', color='white', fontweight='bold'); axes[2].axis('off')
axes[2].set_facecolor('#0D1117')

patches = [mpatches.Patch(color=CLASS_COLORS[k], label=v) for k,v in LAND_COVER_CLASSES.items() if k>0]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10,
           facecolor='#161B22', labelcolor='white', bbox_to_anchor=(0.5,0.0))
plt.suptitle('🛰️ ISRO Land Cover Classification', color='white', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('classification_result.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print('✅ Saved: classification_result.png')

## 🌐 Step 8: Create Interactive Folium Map

In [ ]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles='CartoDB dark_matter')
folium.TileLayer('OpenStreetMap').add_to(m)
folium.TileLayer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Satellite').add_to(m)

folium.Marker(
    [center_lat, center_lon],
    popup=f'<b>Study Area</b><br>Accuracy: {acc*100:.1f}%<br>Classes: {len(LAND_COVER_CLASSES)-1}',
    tooltip='📍 Click for info',
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m)

legend = '''<div style="position:fixed;bottom:30px;right:30px;z-index:9999;
background:rgba(0,0,0,0.85);padding:15px;border-radius:10px;
border:1px solid #00FF88;font-family:monospace">
<b style="color:#00FF88">🛰️ Land Cover</b><br>'''
for k,v in LAND_COVER_CLASSES.items():
    if k==0: continue
    legend += f'<span style="color:{CLASS_COLORS[k]}">■</span> <span style="color:white">{v}</span><br>'
legend += '</div>'
m.get_root().html.add_child(folium.Element(legend))
folium.LayerControl().add_to(m)

m.save('land_cover_map.html')
display(m)
print('✅ Interactive map ready!')

## 💾 Step 9: Save Model & Download Results

In [ ]:
# Save model
joblib.dump({'model': rf, 'scaler': scaler}, 'land_cover_rf_model.pkl')

# Download files from Colab
from google.colab import files
files.download('classification_result.png')
files.download('land_cover_map.html')
files.download('land_cover_rf_model.pkl')

print(f'✅ All done! Accuracy = {acc*100:.2f}%')
print('📁 Files downloaded to your computer!')

---
## 📚 How to Use Real ISRO Data

1. Go to **https://bhuvan.nrsc.gov.in** or **https://bhoonidhi.nrsc.gov.in**
2. Register free → Search your area → Download **LISS-IV** or **Resourcesat-2A** data
3. Upload the `.tif` file to Colab using `files.upload()`
4. Uncomment **Option B** in Step 4 and re-run from there

**Band Order for LISS-IV:** Green (B2), Red (B3), NIR (B4)  
**Band Order for Resourcesat-2:** Green (B2), Red (B3), NIR (B4), SWIR (B5)
